# 00 — Data Fetching

Loads the pre-computed segment feature CSV and enriches it with OSM-derived columns:
`visibility`, `visibility_blocked_ratio`, `public_transport_proximity_m`, and `dominant_land_use_score`.

**Input:** `csv/features_all_boroughs_with_location_id.csv`  
**Output:** `csv/features_all_boroughs_with_location_id_augmented.csv`

**Existing CSV-backed features are preserved as-is.** The visibility column is recomputed from building coverage in a 50 m flat-capped buffer, and the raw blocked-area ratio is stored alongside it.

The land-use column is approximated from building footprints and their ground-level tags, not from parcel landuse polygons.

In [1]:
import sys
from pathlib import Path

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# optional local PBF loader (falls back to Overpass if not available)
try:
    from scripts.building_loader import get_borough_buildings
except Exception as e:
    print(f'  [WARNING] Could not import local building loader: {e}')
    get_borough_buildings = None

ox.settings.log_console = False
ox.settings.use_cache = True

In [2]:
# ── Paths ────────────────────────────────────────────────────────────────────
REPO_ROOT = Path('..').resolve()
CSV_DIR = REPO_ROOT / 'csv'
BASE_FEATURES_CSV = CSV_DIR / 'features_all_boroughs_with_location_id.csv'
AUGMENTED_FEATURES_CSV = CSV_DIR / 'features_all_boroughs_with_location_id_augmented.csv'
MAPILLARY_CSV = CSV_DIR / 'mapillary_streetlights_london_ids.csv'
CRS_PROJECTED = 'EPSG:27700'
# Path to your local Geofabrik `.osm.pbf` file.
PBF_PATH = r'C:\Users\maria\Documents\GitHub\MaCAD26-G01-DataEncoding\geofabrik\greater-london-260529.osm.pbf'
# When True, the notebook will stop if the local PBF path is missing or invalid.
STRICT_LOCAL_BUILDINGS = True

# ── Borough definitions ───────────────────────────────────────────────────────
BOROUGHS = {
    'westminster':    'City of Westminster, London, UK',
    'islington':      'London Borough of Islington, London, UK',
    'hackney':        'London Borough of Hackney, London, UK',
    'tower_hamlets':  'London Borough of Tower Hamlets, London, UK',
    'southwark':      'London Borough of Southwark, London, UK',
}

# Explicit mapping from CSV borough values to query strings
BOROUGH_QUERIES = {v: v for v in BOROUGHS.values()}

# Only keep these boroughs from the input CSV
ALLOWED_BOROUGH_NAMES = [
    'City of Westminster, London, UK',
    'London Borough of Islington, London, UK',
    'London Borough of Hackney, London, UK',
    'London Borough of Tower Hamlets, London, UK',
    'London Borough of Southwark, London, UK',
]

# ── Helper: load and project base feature CSV ────────────────────────────────
def load_base_features(csv_path):
    """Load the pre-computed segment features and attach projected geometry.

    Only rows whose `borough` is in `ALLOWED_BOROUGH_NAMES` are kept.
    """
    base_df = pd.read_csv(csv_path)
    if 'geometry' not in base_df.columns:
        raise ValueError(f'Missing geometry column in {csv_path}')

    # Filter to allowed boroughs
    if 'borough' in base_df.columns:
        base_df = base_df[base_df['borough'].isin(ALLOWED_BOROUGH_NAMES)].reset_index(drop=True)
    else:
        print('  [WARNING] No `borough` column found in base CSV; keeping all rows.')

    geometry = gpd.GeoSeries.from_wkt(base_df['geometry'], crs=CRS_PROJECTED)
    base_df = base_df.drop(columns='geometry')
    return gpd.GeoDataFrame(base_df, geometry=geometry, crs=CRS_PROJECTED)

In [3]:
# ── Helper: load and project Mapillary lights ─────────────────────────────────
def load_mapillary_lights():
    """Load Mapillary CSV and return a projected GeoDataFrame of light points."""
    if not MAPILLARY_CSV.exists():
        print(f'  [WARNING] Mapillary CSV not found at {MAPILLARY_CSV}.')
        print('  lamp_count will be set to NaN for all segments.')
        return None
    df = pd.read_csv(MAPILLARY_CSV)
    # Normalise column names to lowercase
    df.columns = df.columns.str.lower().str.strip()
    # Accept 'lat'/'lon' or 'latitude'/'longitude'
    lat_col = next((c for c in df.columns if c.startswith('lat')), None)
    lon_col = next((c for c in df.columns if c.startswith('lon')), None)
    if lat_col is None or lon_col is None:
        raise ValueError(f'Could not find lat/lon columns in {MAPILLARY_CSV}. Found: {list(df.columns)}')
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs='EPSG:4326'
    ).to_crs(CRS_PROJECTED)
    print(f'  Loaded {len(gdf):,} Mapillary light points.')
    return gdf

In [4]:
# ── Helper: count points within buffer of each segment ────────────────────────
def count_points_in_buffer(edges_proj, points_gdf, buffer_m, col_name):
    """Add a column counting how many points fall within buffer_m of each edge."""
    if points_gdf is None:
        edges_proj[col_name] = np.nan
        return edges_proj
    buffers = edges_proj.geometry.buffer(buffer_m)
    counts = []
    for buf in buffers:
        counts.append(points_gdf[points_gdf.geometry.within(buf)].shape[0])
    edges_proj[col_name] = counts
    return edges_proj

In [5]:
# ── Helper: transit proximity (rail vs bus) ───────────────────────────────────
RAIL_TAGS = {
    'railway': ['subway_entrance', 'station', 'tram_stop'],
    'public_transport': 'stop_position',
}
BUS_TAGS = {
    'highway': 'bus_stop',
    'public_transport': 'platform',
}


def _nearest_dist(edges_proj, stops_gdf, col_name):
    """Add column: distance in metres to nearest stop (EPSG:27700 = metres)."""
    segment_points = edges_proj.geometry.apply(
        lambda geom: geom.interpolate(0.5, normalized=True)
        if geom.geom_type in ('LineString', 'MultiLineString')
        else geom.centroid
    )
    if stops_gdf is None or stops_gdf.empty:
        edges_proj[col_name] = np.nan
        return edges_proj

    stop_coords = np.array([(g.x, g.y) for g in stops_gdf.geometry])
    dists = []
    for mp in segment_points:
        d = np.sqrt(((stop_coords - [mp.x, mp.y]) ** 2).sum(axis=1)).min()
        dists.append(round(float(d), 2))
    edges_proj[col_name] = dists
    return edges_proj


def add_transit_dist(edges_proj, borough_query):
    """
    dist_to_rail_m : nearest tube/overground/DLR/Elizabeth line entrance
    dist_to_bus_m  : nearest bus stop or platform
    """
    def fetch_stops(tags, label):
        try:
            gdf = ox.features_from_place(borough_query, tags=tags)
            gdf['geometry'] = gdf.geometry.apply(
                lambda g: g.centroid if g.geom_type != 'Point' else g
            )
            gdf = gdf[['geometry']].to_crs(CRS_PROJECTED).drop_duplicates('geometry')
            print(f'  {label}: {len(gdf):,} stops found.')
            return gdf
        except Exception as e:
            print(f'  [WARNING] {label} fetch failed: {e}')
            return None

    print('  Fetching rail stops...')
    rail = fetch_stops(RAIL_TAGS, 'rail')
    print('  Fetching bus stops...')
    bus = fetch_stops(BUS_TAGS, 'bus')

    if rail is not None:
        edges_proj = _nearest_dist(edges_proj, rail, 'dist_to_rail_m')
    else:
        edges_proj['dist_to_rail_m'] = np.nan

    if bus is not None:
        edges_proj = _nearest_dist(edges_proj, bus, 'dist_to_bus_m')
    else:
        edges_proj['dist_to_bus_m'] = np.nan

    return edges_proj

In [11]:
# ── Helper: visibility and land-use from building footprints ───────────────────
BUFFER_M = 50
BUFFER_SMALL = BUFFER_M

COMMERCIAL_BUILDING_KEYS = {
    'commercial',
    'retail',
    'industrial',
    'office',
    'warehouse',
    'supermarket',
    'mall',
    'hotel',
    'school',
    'hospital',
    'train_station',
}
COMMERCIAL_BUILDING_VALUES = COMMERCIAL_BUILDING_KEYS
RESIDENTIAL_BUILDING_VALUES = {
    'house',
    'residential',
    'apartments',
    'detached',
    'semi_detached',
    'terrace',
    'dormitory',
    'bungalow',
    'cabin',
    'static_caravan',
}


def _repair_geometries(geoms):
    repaired = []
    for geom in geoms:
        if geom is None or geom.is_empty:
            repaired.append(geom)
            continue
        if not geom.is_valid:
            try:
                geom = geom.buffer(0)
            except Exception:
                pass
        repaired.append(geom)
    return gpd.GeoSeries(repaired, index=geoms.index, crs=geoms.crs)


def load_borough_buildings(borough_query):
    """Fetch borough building polygons once and project them to metres.

    This notebook is configured to avoid the Overpass fallback when `STRICT_LOCAL_BUILDINGS`
    is True. Set `PBF_PATH` to a valid local `.osm.pbf` file before running this notebook.
    """
    if STRICT_LOCAL_BUILDINGS and (PBF_PATH is None or not Path(PBF_PATH).exists()):
        raise FileNotFoundError(
            'Set PBF_PATH to a valid local .osm.pbf file to avoid the Overpass fallback.'
        )

    print('  Loading building footprints from local PBF...')
    if get_borough_buildings is None:
        raise ImportError('scripts.building_loader.get_borough_buildings is not available')

    cache_dir = 'cache/buildings_local' if STRICT_LOCAL_BUILDINGS else 'cache/buildings'
    buildings = get_borough_buildings(
        borough_query,
        pbf_path=PBF_PATH,
        cache_dir=cache_dir,
        allow_overpass_fallback=not STRICT_LOCAL_BUILDINGS,
    )
    if buildings is None or buildings.empty:
        raise ValueError(
            'No building footprints were loaded from the local PBF. Check that PBF_PATH points to the correct extract.'
        )

    buildings = buildings.copy()
    buildings = buildings[buildings.geometry.notna() & ~buildings.geometry.is_empty].copy()
    buildings['geometry'] = _repair_geometries(buildings.geometry)
    buildings = buildings[buildings.geometry.notna() & ~buildings.geometry.is_empty].copy()

    print(f'  building footprints (local/cached): {len(buildings):,} polygons found.')
    return buildings


def _classify_building_row(row):
    building_value = str(row.get('building', '')).strip().lower()
    commercial = building_value in COMMERCIAL_BUILDING_VALUES or building_value in COMMERCIAL_BUILDING_KEYS
    residential = building_value in RESIDENTIAL_BUILDING_VALUES
    if commercial and residential:
        return 0.5
    if commercial:
        return 1.0
    if residential:
        return 0.0
    return 0.5


def _visibility_blockage_ratio_in_buffer(buildings_gdf, buffer_geom):
    candidates = buildings_gdf[buildings_gdf.geometry.intersects(buffer_geom)]
    if candidates.empty:
        return 0.0

    try:
        intersections = candidates.geometry.intersection(buffer_geom)
    except Exception:
        repaired = candidates.copy()
        repaired['geometry'] = _repair_geometries(repaired.geometry)
        repaired = repaired[repaired.geometry.notna() & ~repaired.geometry.is_empty]
        if repaired.empty:
            return 0.0
        intersections = repaired.geometry.intersection(buffer_geom)

    covered_geom = intersections.unary_union
    buffer_area = buffer_geom.area
    if buffer_area == 0:
        return 0.0

    blocked_ratio = float(covered_geom.area / buffer_area)
    return round(float(np.clip(blocked_ratio, 0.0, 1.0)), 4)


def add_visibility_from_buildings(edges_proj, buildings_gdf, buffer_m=BUFFER_M):
    """Add raw blockage ratio and openness visibility from building coverage."""
    if buildings_gdf is None:
        edges_proj['visibility_blocked_ratio'] = np.nan
        edges_proj['visibility'] = np.nan
        return edges_proj

    blocked_ratios = []
    visibility_scores = []
    for geom in edges_proj.geometry:
        buffer_geom = geom.buffer(buffer_m, cap_style=2)
        blocked_ratio = _visibility_blockage_ratio_in_buffer(buildings_gdf, buffer_geom)
        visibility_score = round(float(1.0 - blocked_ratio), 4)
        blocked_ratios.append(blocked_ratio)
        visibility_scores.append(visibility_score)

    edges_proj['visibility_blocked_ratio'] = blocked_ratios
    edges_proj['visibility'] = visibility_scores
    return edges_proj


def _area_weighted_score_in_buffer(buildings_gdf, buffer_geom):
    candidates = buildings_gdf[buildings_gdf.geometry.intersects(buffer_geom)]
    if candidates.empty:
        return None

    try:
        intersections = candidates.geometry.intersection(buffer_geom)
    except Exception:
        repaired = candidates.copy()
        repaired['geometry'] = _repair_geometries(repaired.geometry)
        repaired = repaired[repaired.geometry.notna() & ~repaired.geometry.is_empty]
        if repaired.empty:
            return None
        intersections = repaired.geometry.intersection(buffer_geom)

    areas = intersections.area
    total_area = areas.sum()
    if total_area == 0:
        return None

    commercial_area = 0.0
    residential_area = 0.0
    for area, score in zip(areas, candidates['building_use_score']):
        commercial_area += area * score
        residential_area += area * (1.0 - score)

    raw = (commercial_area - residential_area) / total_area
    mapped = (raw + 1.0) / 2.0
    return round(float(mapped), 4)


def add_dominant_land_use_score(edges_proj, borough_query, buildings_gdf=None):
    """
    dominant_land_use_score:
     0.0 = residential-dominant buildings
     0.5 = balanced or no usable building-use signal
     1.0 = commercial-dominant buildings
    """
    if buildings_gdf is None:
        edges_proj['dominant_land_use_score'] = np.nan
        return edges_proj

    buildings = buildings_gdf
    if 'building_use_score' not in buildings.columns:
        buildings = buildings.copy()
        buildings['building_use_score'] = buildings.apply(_classify_building_row, axis=1)

    scores = []
    for geom in edges_proj.geometry:
        buf = geom.buffer(BUFFER_M, cap_style=2)
        score = _area_weighted_score_in_buffer(buildings, buf)
        scores.append(np.nan if score is None else score)

    edges_proj['dominant_land_use_score'] = scores
    return edges_proj

In [7]:
# ── Helper: residential flag ──────────────────────────────────────────────────
def add_residential_flag(edges_proj, borough_query):
    """1 if a residential building footprint overlaps the buffer, else 0."""
    print('  Loading building footprints for residential flag...')
    # reuse loader which may be cached/local; fallback is disabled in strict mode
    buildings = None
    try:
        if get_borough_buildings is not None:
            cache_dir = 'cache/buildings_local' if STRICT_LOCAL_BUILDINGS else 'cache/buildings'
            buildings = get_borough_buildings(
                borough_query,
                pbf_path=PBF_PATH,
                cache_dir=cache_dir,
                allow_overpass_fallback=not STRICT_LOCAL_BUILDINGS,
            )
        else:
            raise ImportError('scripts.building_loader.get_borough_buildings is not available')
    except Exception as e:
        print(f'  [WARNING] Could not fetch building footprints: {e}')
        edges_proj['residential_flag'] = np.nan
        return edges_proj

    if buildings is None or buildings.empty:
        edges_proj['residential_flag'] = np.nan
        return edges_proj

    buildings['is_residential'] = buildings['building'].astype(str).str.lower().isin(RESIDENTIAL_BUILDING_VALUES)
    flags = []
    for geom in edges_proj.geometry:
        buf = geom.buffer(BUFFER_SMALL)
        flag = int(buildings[buildings['is_residential'] & buildings.geometry.intersects(buf)].shape[0] > 0)
        flags.append(flag)
    edges_proj['residential_flag'] = flags
    return edges_proj

In [8]:
# ── Helper: transit proximity (rail vs bus) ───────────────────────────────────
RAIL_TAGS = {
    'railway': ['subway_entrance', 'station', 'tram_stop'],
    'public_transport': 'stop_position',
}
BUS_TAGS = {
    'highway': 'bus_stop',
    'public_transport': 'platform',
}


def _nearest_dist(edges_proj, stops_gdf, col_name):
    """Add column: distance in metres to nearest stop (EPSG:27700 = metres)."""
    segment_points = edges_proj.geometry.apply(
        lambda geom: geom.interpolate(0.5, normalized=True)
        if geom.geom_type in ('LineString', 'MultiLineString')
        else geom.centroid
    )
    if stops_gdf is None or stops_gdf.empty:
        edges_proj[col_name] = np.nan
        return edges_proj

    stop_coords = np.array([(g.x, g.y) for g in stops_gdf.geometry])
    dists = []
    for mp in segment_points:
        d = np.sqrt(((stop_coords - [mp.x, mp.y]) ** 2).sum(axis=1)).min()
        dists.append(round(float(d), 2))
    edges_proj[col_name] = dists
    return edges_proj


def add_transit_dist(edges_proj, borough_query):
    """
    dist_to_rail_m : nearest tube/overground/DLR/Elizabeth line entrance
    dist_to_bus_m  : nearest bus stop or platform
    """
    def fetch_stops(tags, label):
        try:
            gdf = ox.features_from_place(borough_query, tags=tags)
            gdf['geometry'] = gdf.geometry.apply(
                lambda g: g.centroid if g.geom_type != 'Point' else g
            )
            gdf = gdf[['geometry']].to_crs(CRS_PROJECTED).drop_duplicates('geometry')
            print(f'  {label}: {len(gdf):,} stops found.')
            return gdf
        except Exception as e:
            print(f'  [WARNING] {label} fetch failed: {e}')
            return None

    print('  Fetching rail stops...')
    rail = fetch_stops(RAIL_TAGS, 'rail')
    print('  Fetching bus stops...')
    bus = fetch_stops(BUS_TAGS, 'bus')

    if rail is not None:
        edges_proj = _nearest_dist(edges_proj, rail, 'dist_to_rail_m')
    else:
        edges_proj['dist_to_rail_m'] = np.nan

    if bus is not None:
        edges_proj = _nearest_dist(edges_proj, bus, 'dist_to_bus_m')
    else:
        edges_proj['dist_to_bus_m'] = np.nan

    return edges_proj

In [12]:
# ── Main enrichment loop ─────────────────────────────────────────────────────
if STRICT_LOCAL_BUILDINGS and (PBF_PATH is None or not Path(PBF_PATH).exists()):
    raise FileNotFoundError(
        'Set PBF_PATH to a valid local .osm.pbf file before running this notebook. '\
        'The notebook is configured to avoid the Overpass fallback.'
    )

base_features = load_base_features(BASE_FEATURES_CSV)
base_features['_row_order'] = np.arange(len(base_features))
base_features['visibility'] = np.nan
base_features['visibility_blocked_ratio'] = np.nan
base_features['public_transport_proximity_m'] = np.nan
base_features['dominant_land_use_score'] = np.nan

for borough_name, borough_features in base_features.groupby('borough', sort=False):
    borough_query = BOROUGH_QUERIES.get(borough_name)
    if borough_query is None:
        print(f'\n[WARNING] No OSM query configured for {borough_name!r}; leaving new columns as NaN.')
        continue

    print(f'\n══ {borough_name.upper()} ══')
    working = borough_features.copy()
    buildings = load_borough_buildings(borough_query)
    working = add_visibility_from_buildings(working, buildings)
    working = add_transit_dist(working, borough_query)
    working['public_transport_proximity_m'] = working[['dist_to_rail_m', 'dist_to_bus_m']].min(axis=1)
    working = working.drop(columns=['dist_to_rail_m', 'dist_to_bus_m'])

    working = add_dominant_land_use_score(working, borough_query, buildings_gdf=buildings)

    base_features.loc[working.index, 'visibility'] = working['visibility']
    base_features.loc[working.index, 'visibility_blocked_ratio'] = working['visibility_blocked_ratio']
    base_features.loc[working.index, 'public_transport_proximity_m'] = working['public_transport_proximity_m']
    base_features.loc[working.index, 'dominant_land_use_score'] = working['dominant_land_use_score']

base_features = base_features.sort_values('_row_order').drop(columns='_row_order')

if 'geometry' in base_features.columns:
    base_features['geometry'] = base_features.geometry.to_wkt()

AUGMENTED_FEATURES_CSV.parent.mkdir(parents=True, exist_ok=True)
base_features.to_csv(AUGMENTED_FEATURES_CSV, index=False)
print(f'\nSaved → {AUGMENTED_FEATURES_CSV.name}  ({len(base_features):,} rows)')


══ LONDON BOROUGH OF ISLINGTON, LONDON, UK ══
  Loading building footprints from local PBF...
  building footprints (local/cached): 41,473 polygons found.
  Fetching rail stops...
  rail: 406 stops found.
  Fetching bus stops...
  bus: 442 stops found.

══ LONDON BOROUGH OF TOWER HAMLETS, LONDON, UK ══
  Loading building footprints from local PBF...
  building footprints (local/cached): 36,339 polygons found.
  Fetching rail stops...
  rail: 607 stops found.
  Fetching bus stops...
  bus: 565 stops found.

══ LONDON BOROUGH OF HACKNEY, LONDON, UK ══
  Loading building footprints from local PBF...
  building footprints (local/cached): 40,721 polygons found.
  Fetching rail stops...
  rail: 370 stops found.
  Fetching bus stops...
  bus: 511 stops found.

══ LONDON BOROUGH OF SOUTHWARK, LONDON, UK ══
  Loading building footprints from local PBF...
  building footprints (local/cached): 59,080 polygons found.
  Fetching rail stops...
  rail: 457 stops found.
  Fetching bus stops...
  bus:

In [13]:
# ── Validation summary ────────────────────────────────────────────────────────
print('\n── Augmented data summary ──')
if AUGMENTED_FEATURES_CSV.exists():
    df = pd.read_csv(AUGMENTED_FEATURES_CSV)
    print(f'Loaded {len(df):,} rows from {AUGMENTED_FEATURES_CSV.name}')
    print(f"Columns: {', '.join(df.columns)}")
    coverage_cols = ['visibility', 'visibility_blocked_ratio', 'public_transport_proximity_m', 'dominant_land_use_score']
    available_cols = [col for col in coverage_cols if col in df.columns]
    print('\nCoverage summary:')
    print(df[available_cols].describe().T[['count', 'mean', 'min', 'max']].to_string())
    nan_pct = df[available_cols].isna().mean().mul(100).round(1)
    print('\nMissing values in new columns (%):')
    print(nan_pct.to_string())
else:
    print(f'{AUGMENTED_FEATURES_CSV} not found')


── Augmented data summary ──
Loaded 35,978 rows from features_all_boroughs_with_location_id_augmented.csv
Columns: lighting, visibility, connectivity, enclosure, borough, latitude, longitude, location_id, geometry, visibility_blocked_ratio, public_transport_proximity_m, dominant_land_use_score

Coverage summary:
                                count        mean     min       max
visibility                    35978.0    0.703532  0.2468    1.0000
visibility_blocked_ratio      35978.0    0.296468  0.0000    0.7532
public_transport_proximity_m  35978.0  110.696102  0.4500  541.2400
dominant_land_use_score       35903.0    0.309621  0.0000    1.0000

Missing values in new columns (%):
visibility                      0.0
visibility_blocked_ratio        0.0
public_transport_proximity_m    0.0
dominant_land_use_score         0.2
